In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

import joblib

In [2]:
df = pd.read_csv("mfcc_dataset.csv")

X = df.iloc[:, 1:].values.astype(np.float32)
y = df.iloc[:, 0].values

print("Dataset shape:", X.shape)
print("Labels:", np.unique(y))

Dataset shape: (7, 13)
Labels: ['a' 'b' 'c' 'd' 'e']


In [3]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

num_classes = len(np.unique(y_encoded))

print("Classes:", label_encoder.classes_)

Classes: ['a' 'b' 'c' 'd' 'e']


Normalize Features


In [4]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

Train/Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_encoded,
    test_size=0.2,
    random_state=42
)

In [6]:
class MFCCDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [7]:
train_dataset = MFCCDataset(X_train, y_train)
test_dataset = MFCCDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

Neural Network


In [8]:
class MFCCNet(nn.Module):

    def __init__(self, input_size=13, num_classes=3):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.model(x)


model = MFCCNet(input_size=13, num_classes=num_classes)

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [10]:
epochs = 50

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss:.4f}")

Epoch 1/50 | Loss: 1.5895
Epoch 2/50 | Loss: 1.5661
Epoch 3/50 | Loss: 1.5433
Epoch 4/50 | Loss: 1.5218
Epoch 5/50 | Loss: 1.5019
Epoch 6/50 | Loss: 1.4820
Epoch 7/50 | Loss: 1.4632
Epoch 8/50 | Loss: 1.4443
Epoch 9/50 | Loss: 1.4254
Epoch 10/50 | Loss: 1.4062
Epoch 11/50 | Loss: 1.3871
Epoch 12/50 | Loss: 1.3679
Epoch 13/50 | Loss: 1.3486
Epoch 14/50 | Loss: 1.3289
Epoch 15/50 | Loss: 1.3089
Epoch 16/50 | Loss: 1.2886
Epoch 17/50 | Loss: 1.2688
Epoch 18/50 | Loss: 1.2486
Epoch 19/50 | Loss: 1.2283
Epoch 20/50 | Loss: 1.2078
Epoch 21/50 | Loss: 1.1872
Epoch 22/50 | Loss: 1.1662
Epoch 23/50 | Loss: 1.1450
Epoch 24/50 | Loss: 1.1235
Epoch 25/50 | Loss: 1.1018
Epoch 26/50 | Loss: 1.0797
Epoch 27/50 | Loss: 1.0573
Epoch 28/50 | Loss: 1.0350
Epoch 29/50 | Loss: 1.0124
Epoch 30/50 | Loss: 0.9896
Epoch 31/50 | Loss: 0.9666
Epoch 32/50 | Loss: 0.9434
Epoch 33/50 | Loss: 0.9201
Epoch 34/50 | Loss: 0.8969
Epoch 35/50 | Loss: 0.8736
Epoch 36/50 | Loss: 0.8499
Epoch 37/50 | Loss: 0.8261
Epoch 38/5

In [11]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        outputs = model(X_batch)
        preds = torch.argmax(outputs, dim=1)

        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

print("Accuracy:", correct / total)

Accuracy: 0.5


In [12]:
torch.save(model.state_dict(), "mfcc_model.pth")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")

print("Model saved successfully.")

Model saved successfully.


In [25]:
def predict(sample):

    sample = scaler.transform([sample]).astype(np.float32)
    sample = torch.tensor(sample, dtype=torch.float32)

    with torch.no_grad():
        output = model(sample)
        pred = torch.argmax(output, dim=1).item()

    return label_encoder.inverse_transform([pred])[0]

In [26]:
import torch
import numpy as np
import joblib

import DCT  # your custom MFCC generator


# ==========================================================
# LOAD TRAINED MODEL + PREPROCESSORS
# ==========================================================

model = MFCCNet(input_size=13, num_classes=num_classes)
model.load_state_dict(torch.load("mfcc_model.pth"))
model.eval()

scaler = joblib.load("scaler.pkl")
label_encoder = joblib.load("label_encoder.pkl")

predict using dct.py mfcc

In [33]:
def predict_from_dct():

    # ======================================================
    # 1. Run your MFCC pipeline (DCT.py already handles audio)
    # ======================================================
    MFCCS = DCT.MFCCS   # shape: (frames, 13)
    
    

    # ======================================================
    # 2. Convert to feature vector (same as training)
    # ======================================================
    sample = MFCCS.mean(axis=0).astype(np.float32)
    print("Feature vector is:", sample)

    # ======================================================
    # 3. Normalize using TRAINED scaler
    # ======================================================
    sample = scaler.transform([sample]).astype(np.float32)

    # ======================================================
    # 4. Convert to tensor
    # ======================================================
    sample = torch.tensor(sample, dtype=torch.float32)

    # ======================================================
    # 5. Predict
    # ======================================================
    with torch.no_grad():
        output = model(sample)
        pred = torch.argmax(output, dim=1).item()

    return label_encoder.inverse_transform([pred])[0]

In [31]:
import importlib
import DCT

importlib.reload(DCT)

<module 'DCT' from '/home/sas/Desktop/Git_hub/mfcc-speech-recognition/MFCC/DCT.py'>

In [35]:
result = predict_from_dct()

print("Predicted Label:", result)

Feature vector is: [-42.893723   -19.979218   -17.439486     9.136627    11.597967
  23.57596     -0.04426254  -4.822558     0.99246454  -1.1207505
   7.359239     1.9362093    7.4281645 ]
Predicted Label: a


In [ ]:
print(list(model.parameters())[0][0][:5])

tensor([-0.0294,  0.0160, -0.0192, -0.2020,  0.0109], grad_fn=<SliceBackward0>)
